In [28]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

from joblib import load
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

In [30]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]

BASE_DIR = PROJECT_ROOT / "data" / "datasets"
MODEL_DIR = PROJECT_ROOT / "models"

logreg = load(MODEL_DIR / "logreg_grid.pkl")
rf     = load(MODEL_DIR / "rf_grid.pkl")
xgb    = load(MODEL_DIR / "xgb_grid.pkl")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_DIR:", BASE_DIR)
print("MODEL_DIR:", MODEL_DIR)

<class 'sklearn.pipeline.Pipeline'>
<class 'sklearn.pipeline.Pipeline'>
<class 'dict'>


In [31]:
THEFT_CODES = {"23A", "23B", "23C", "23D", "23E", "23F", "23G", "23H"}

FEATURE_COLS = [
    "cell_id",
    "year", "month", "weekofyear", "dayofweek",
    "lag_1", "lag_2", "lag_4",
    "roll_mean_4", "roll_sum_4",
    "roll_mean_8", "roll_sum_8", "roll_std_8"
]

def find_col(cols, candidates):
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for c in cols:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    raise KeyError(f"Cannot find column among {candidates}. Available: {list(cols)[:30]}")

def build_events_from_tables(table_dir: Path):
    incident_path = table_dir / "NIBRS_incident.csv"
    offense_path  = table_dir / "NIBRS_OFFENSE.csv"
    offtype_path  = table_dir / "NIBRS_OFFENSE_TYPE.csv"

    offense = pd.read_csv(offense_path, low_memory=False)
    offtype = pd.read_csv(offtype_path, low_memory=False)

    col_off_incident = find_col(offense.columns, ["incident_id"])

    # 新结构：OFFENSE 里直接有 offense_code
    if "offense_code" in [c.lower() for c in offense.columns]:
        col_off_code = find_col(offense.columns, ["offense_code"])
        offense[col_off_code] = offense[col_off_code].astype(str)
        theft_off = offense[offense[col_off_code].isin(THEFT_CODES)].copy()

    # 老结构：OFFENSE 里只有 offense_type_id
    else:
        col_off_typeid = find_col(offense.columns, ["offense_type_id"])
        col_offtype_id = find_col(offtype.columns, ["offense_type_id"])
        col_offtype_code = find_col(offtype.columns, ["offense_code"])

        offense[col_off_typeid] = pd.to_numeric(offense[col_off_typeid], errors="coerce")
        offtype[col_offtype_id] = pd.to_numeric(offtype[col_offtype_id], errors="coerce")

        offense = offense.dropna(subset=[col_off_typeid]).copy()
        offtype = offtype.dropna(subset=[col_offtype_id]).copy()

        offense[col_off_typeid] = offense[col_off_typeid].astype(int)
        offtype[col_offtype_id] = offtype[col_offtype_id].astype(int)

        offense2 = offense.merge(
            offtype[[col_offtype_id, col_offtype_code]],
            left_on=col_off_typeid,
            right_on=col_offtype_id,
            how="left"
        )
        offense2[col_offtype_code] = offense2[col_offtype_code].astype(str)
        theft_off = offense2[offense2[col_offtype_code].isin(THEFT_CODES)].copy()

    theft_incidents = theft_off[[col_off_incident]].drop_duplicates()

    incident = pd.read_csv(incident_path, low_memory=False)
    col_inc_incident = find_col(incident.columns, ["incident_id"])
    col_inc_agency   = find_col(incident.columns, ["agency_id"])
    col_inc_date     = find_col(incident.columns, ["incident_date", "date"])

    events = theft_incidents.merge(
        incident[[col_inc_incident, col_inc_agency, col_inc_date]],
        left_on=col_off_incident,
        right_on=col_inc_incident,
        how="inner"
    )

    events = events.rename(columns={
        col_inc_incident: "incident_id",
        col_inc_agency:   "cell_id",
        col_inc_date:     "date"
    })

    events["date"] = pd.to_datetime(events["date"], errors="coerce")
    events = events.dropna(subset=["incident_id", "cell_id", "date"]).copy()
    events = events.drop_duplicates(subset=["incident_id"]).copy()
    events["cell_id"] = events["cell_id"].astype(str)

    return events

def build_weekly_panel(events_df):
    df = events_df.copy()
    df["time_bin"] = df["date"].dt.to_period("W").dt.start_time

    panel = (
        df.groupby(["cell_id", "time_bin"], as_index=False)
          .size()
          .rename(columns={"size": "crime_count"})
    )

    cells = panel["cell_id"].unique()
    tmin, tmax = panel["time_bin"].min(), panel["time_bin"].max()
    full_time = pd.date_range(tmin, tmax, freq="7D")

    full_index = pd.MultiIndex.from_product(
        [cells, full_time],
        names=["cell_id", "time_bin"]
    )

    panel = (
        panel.set_index(["cell_id", "time_bin"])
             .reindex(full_index)
             .fillna(0)
             .reset_index()
    )

    panel["crime_count"] = panel["crime_count"].astype(int)
    return panel

def make_features(panel_df):
    out = panel_df.sort_values(["cell_id", "time_bin"]).copy()
    g = out.groupby("cell_id", group_keys=False)

    out["y_next_count"] = g["crime_count"].shift(-1)
    out = out.dropna(subset=["y_next_count"]).copy()
    out["y"] = (out["y_next_count"] > 0).astype(int)

    dt = pd.to_datetime(out["time_bin"])
    out["year"] = dt.dt.year
    out["month"] = dt.dt.month
    out["weekofyear"] = dt.dt.isocalendar().week.astype(int)
    out["dayofweek"] = dt.dt.dayofweek

    out["lag_1"] = g["crime_count"].shift(1)
    out["lag_2"] = g["crime_count"].shift(2)
    out["lag_4"] = g["crime_count"].shift(4)

    base = g["crime_count"].shift(1)
    out["roll_mean_4"] = base.rolling(4).mean()
    out["roll_sum_4"]  = base.rolling(4).sum()
    out["roll_mean_8"] = base.rolling(8).mean()
    out["roll_sum_8"]  = base.rolling(8).sum()
    out["roll_std_8"]  = base.rolling(8).std()

    fill_cols = [
        "lag_1", "lag_2", "lag_4",
        "roll_mean_4", "roll_sum_4",
        "roll_mean_8", "roll_sum_8", "roll_std_8"
    ]
    out[fill_cols] = out[fill_cols].fillna(0.0)

    return out

def make_X_routeB(feat_df):
    X = feat_df[FEATURE_COLS].copy()
    for c in FEATURE_COLS:
        if c != "cell_id":
            X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0.0)

    # Route B 核心：只 groupby，不让区域ID进入模型
    X["cell_id"] = "ALL"
    return X

def predict_all_models(feat_df, X_df):
    out = feat_df.copy()

    out["risk_logreg"] = logreg.predict_proba(X_df)[:, 1]
    out["risk_rf"]     = rf.predict_proba(X_df)[:, 1]

    X_xgb = xgb["preprocess"].transform(X_df)
    out["risk_xgb"] = xgb["model"].predict_proba(X_xgb)[:, 1]

    return out

def evaluate_overall(feat_df):
    y_true = feat_df["y"].values
    pos_rate = float(y_true.mean())

    res = {
        "positive_rate": pos_rate,
        "n_rows": int(len(feat_df)),
        "n_pos": int(y_true.sum()),
    }

    for name in ["logreg", "rf", "xgb"]:
        y_score = feat_df[f"risk_{name}"].values
        res[f"roc_auc_{name}"] = float(roc_auc_score(y_true, y_score))
        res[f"pr_auc_{name}"]  = float(average_precision_score(y_true, y_score))

    return res

def evaluate_time_split(feat_df):
    df = feat_df.copy()
    df["_t"] = pd.to_datetime(df["time_bin"])

    weeks = np.array(sorted(df["_t"].unique()))
    n_weeks = len(weeks)
    cut_val = weeks[int(n_weeks * 0.70)]

    val_mask  = df["_t"] < cut_val
    test_mask = df["_t"] >= cut_val

    y_val  = df.loc[val_mask, "y"].values
    y_test = df.loc[test_mask, "y"].values

    thresholds = np.linspace(0.05, 0.95, 19)

    out = {
        "n_weeks": int(n_weeks),
        "val_pos_rate": float(y_val.mean()) if len(y_val) else np.nan,
        "test_pos_rate": float(y_test.mean()) if len(y_test) else np.nan,
        "cut_val_date": str(pd.Timestamp(cut_val).date()),
    }

    for name in ["logreg", "rf", "xgb"]:
        s_val  = df.loc[val_mask,  f"risk_{name}"].values
        s_test = df.loc[test_mask, f"risk_{name}"].values

        auc = roc_auc_score(y_test, s_test)
        ap  = average_precision_score(y_test, s_test)

        best_f1, best_t = -1.0, None
        for t in thresholds:
            pred_val = (s_val >= t).astype(int)
            f1 = f1_score(y_val, pred_val, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)

        pred_test = (s_test >= best_t).astype(int)
        prec = precision_score(y_test, pred_test, zero_division=0)
        rec  = recall_score(y_test, pred_test, zero_division=0)
        f1t  = f1_score(y_test, pred_test, zero_division=0)
        cm   = confusion_matrix(y_test, pred_test)

        out[f"test_roc_auc_{name}"] = float(auc)
        out[f"test_pr_auc_{name}"]  = float(ap)
        out[f"best_threshold_{name}"] = float(best_t)
        out[f"test_precision_{name}"] = float(prec)
        out[f"test_recall_{name}"] = float(rec)
        out[f"test_f1_{name}"] = float(f1t)
        out[f"cm_{name}"] = cm.tolist()

    return out

def process_one_dataset(tag, dataset_name):
    table_dir = BASE_DIR / tag / dataset_name / "tables"

    events = build_events_from_tables(table_dir)
    panel  = build_weekly_panel(events)
    feat   = make_features(panel)
    X      = make_X_routeB(feat)
    feat   = predict_all_models(feat, X)

    overall = evaluate_overall(feat)
    split   = evaluate_time_split(feat)

    result = {
        "tag": tag,
        "dataset": dataset_name,
        "n_events": int(len(events)),
        "n_cells": int(events["cell_id"].nunique()),
        "panel_rows": int(len(panel)),
        "feat_rows": int(len(feat)),
    }
    result.update(overall)
    result.update(split)

    return feat, result

In [32]:
TEMPORAL_DATASETS = ["IL_2021", "IL_2022", "IL_2023", "IL_2024"]
GEOGRAPHIC_DATASETS = ["IL_2024", "IN_2024", "MI_2024", "CA_2024", "NY_2024"]

all_jobs = [("temporal", ds) for ds in TEMPORAL_DATASETS] + \
           [("geographic", ds) for ds in GEOGRAPHIC_DATASETS]

all_jobs

[('temporal', 'IL_2021'),
 ('temporal', 'IL_2022'),
 ('temporal', 'IL_2023'),
 ('temporal', 'IL_2024'),
 ('geographic', 'IL_2024'),
 ('geographic', 'IN_2024'),
 ('geographic', 'MI_2024'),
 ('geographic', 'CA_2024'),
 ('geographic', 'NY_2024')]

In [33]:
all_results = []
all_feats = {}
all_errors = []

for tag, dataset_name in all_jobs:
    print(f"\n===== Running {tag} / {dataset_name} =====")
    try:
        feat_out, result = process_one_dataset(tag, dataset_name)
        all_feats[(tag, dataset_name)] = feat_out
        all_results.append(result)

        print(
            f"Done: {tag}/{dataset_name} | "
            f"rows={result['feat_rows']} | "
            f"logreg AUC={result['roc_auc_logreg']:.4f}, "
            f"rf AUC={result['roc_auc_rf']:.4f}, "
            f"xgb AUC={result['roc_auc_xgb']:.4f}"
        )
    except Exception as e:
        all_errors.append({"tag": tag, "dataset": dataset_name, "error": repr(e)})
        print(f"[FAIL] {tag}/{dataset_name} -> {repr(e)}")


===== Running temporal / IL_2021 =====
Done: temporal/IL_2021 | rows=15548 | logreg AUC=0.8750, rf AUC=0.8827, xgb AUC=0.8809

===== Running temporal / IL_2022 =====
Done: temporal/IL_2022 | rows=24284 | logreg AUC=0.8758, rf AUC=0.8831, xgb AUC=0.8853

===== Running temporal / IL_2023 =====
Done: temporal/IL_2023 | rows=28912 | logreg AUC=0.8511, rf AUC=0.8627, xgb AUC=0.8628

===== Running temporal / IL_2024 =====
Done: temporal/IL_2024 | rows=31096 | logreg AUC=0.8492, rf AUC=0.8699, xgb AUC=0.8709

===== Running geographic / IL_2024 =====
Done: geographic/IL_2024 | rows=31096 | logreg AUC=0.8492, rf AUC=0.8699, xgb AUC=0.8709

===== Running geographic / IN_2024 =====
Done: geographic/IN_2024 | rows=11388 | logreg AUC=0.8626, rf AUC=0.8792, xgb AUC=0.8815

===== Running geographic / MI_2024 =====
Done: geographic/MI_2024 | rows=30316 | logreg AUC=0.8289, rf AUC=0.8484, xgb AUC=0.8483

===== Running geographic / CA_2024 =====
Done: geographic/CA_2024 | rows=29328 | logreg AUC=0.9025

In [34]:
results_df = pd.DataFrame(all_results)

show_cols = [
    "tag", "dataset",
    "n_events", "n_cells", "panel_rows", "feat_rows",
    "positive_rate",
    "roc_auc_logreg", "pr_auc_logreg",
    "roc_auc_rf", "pr_auc_rf",
    "roc_auc_xgb", "pr_auc_xgb",
]

results_df[show_cols].sort_values(["tag", "dataset"]).reset_index(drop=True)

,tag,dataset,n_events,n_cells,panel_rows,feat_rows,positive_rate,roc_auc_logreg,pr_auc_logreg,roc_auc_rf,pr_auc_rf,roc_auc_xgb,pr_auc_xgb
0,geographic,CA_2024,352478,564,29892,29328,0.704583,0.902512,0.962085,0.907430,0.962687,0.916797,0.966136
1,geographic,IL_2024,139517,598,31694,31096,0.520774,0.849187,0.881640,0.869942,0.892104,0.870862,0.893520
2,geographic,IN_2024,64408,219,11607,11388,0.565683,0.862574,0.906315,0.879216,0.913474,0.881524,0.915421
3,geographic,MI_2024,96875,583,30899,30316,0.498120,0.828933,0.860054,0.848410,0.869354,0.848337,0.871461
4,geographic,NY_2024,239257,191,10123,9932,0.686569,0.872270,0.944823,0.881851,0.947167,0.887664,0.949736
5,temporal,IL_2021,59994,299,15847,15548,0.464497,0.874995,0.880613,0.882708,0.885874,0.880944,0.885471
6,temporal,IL_2022,122476,467,24751,24284,0.510171,0.875774,0.899075,0.883107,0.903571,0.885293,0.905643
7,temporal,IL_2023,134812,556,29468,28912,0.547489,0.851119,0.893973,0.862655,0.899082,0.862835,0.901021
8,temporal,IL_2024,139517,598,31694,31096,0.520774,0.849187,0.881640,0.869942,0.892104,0.870862,0.893520


In [35]:
test_cols = [
    "tag", "dataset",
    "cut_val_date",
    "test_roc_auc_logreg", "test_pr_auc_logreg", "best_threshold_logreg", "test_f1_logreg",
    "test_roc_auc_rf", "test_pr_auc_rf", "best_threshold_rf", "test_f1_rf",
    "test_roc_auc_xgb", "test_pr_auc_xgb", "best_threshold_xgb", "test_f1_xgb",
]

results_df[test_cols].sort_values(["tag", "dataset"]).reset_index(drop=True)

,tag,dataset,cut_val_date,test_roc_auc_logreg,test_pr_auc_logreg,best_threshold_logreg,test_f1_logreg,test_roc_auc_rf,test_pr_auc_rf,best_threshold_rf,test_f1_rf,test_roc_auc_xgb,test_pr_auc_xgb,best_threshold_xgb,test_f1_xgb
0,geographic,CA_2024,2024-09-09,0.931795,0.970292,0.5,0.900242,0.934587,0.971656,0.40,0.899601,0.932621,0.969902,0.50,0.906995
1,geographic,IL_2024,2024-09-09,0.886024,0.897543,0.5,0.810311,0.889373,0.901004,0.40,0.812600,0.888627,0.899039,0.45,0.816709
2,geographic,IN_2024,2024-09-09,0.893696,0.916797,0.5,0.830415,0.895275,0.918271,0.40,0.829396,0.894300,0.916457,0.45,0.833169
3,geographic,MI_2024,2024-09-09,0.869130,0.878251,0.5,0.771813,0.872832,0.881972,0.40,0.770721,0.870684,0.879782,0.45,0.777289
4,geographic,NY_2024,2024-09-09,0.913892,0.956240,0.5,0.873323,0.916343,0.958216,0.40,0.871702,0.914124,0.955031,0.50,0.879678
5,temporal,IL_2021,2021-09-06,0.903743,0.915311,0.5,0.828462,0.905358,0.919119,0.35,0.818994,0.904045,0.917147,0.40,0.828155
6,temporal,IL_2022,2022-09-05,0.900419,0.919104,0.5,0.828882,0.898484,0.919475,0.40,0.827837,0.900164,0.919299,0.35,0.819839
7,temporal,IL_2023,2023-09-04,0.895941,0.917153,0.5,0.825575,0.896771,0.917896,0.40,0.825716,0.896033,0.917456,0.40,0.826150
8,temporal,IL_2024,2024-09-09,0.886024,0.897543,0.5,0.810311,0.889373,0.901004,0.40,0.812600,0.888627,0.899039,0.45,0.816709
